# Synthetic full-vocabulary experiments

Two controlled experiments stress-test the revised independent concept-presence model with **2,347 concept slots**:

1. Chest trauma severity stratification (~25.3% positive).
2. ICU in-hospital mortality prediction (~5.2% positive).

Task-relevant slots use real three-character ICD-10 categories. The remaining slots are neutral synthetic distractors; the clinical notebook separately loads the complete real vocabulary from `df_icd10.csv`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "synthetic_two_tasks_2347"
SCRIPT = ROOT / "run_two_task_synthetic_2347.py"
assert DATA_DIR.exists() and SCRIPT.exists()

## Optional rerun

The included results use three model seeds. Set `RUN_EXPERIMENTS=True` to regenerate both datasets and rerun outcome-only versus note-supervised training.

In [2]:
RUN_EXPERIMENTS = False

if RUN_EXPERIMENTS:
    subprocess.run([sys.executable, str(SCRIPT)], check=True)

In [3]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
vocabulary = pd.read_csv(DATA_DIR / "synthetic_vocabulary_2347.csv")

assert len(vocabulary) == 2347
manifest

{'num_concepts': 2347,
 'vocabulary_file': 'synthetic_vocabulary_2347.csv',
 'tasks': {'chest_trauma': {'train': {'n': 900,
    'positive_n': 228,
    'positive_rate': 0.25333333333333335,
    'mean_present_concepts': 10.493333333333334,
    'min_present_concepts': 4,
    'max_present_concepts': 19},
   'dev': {'n': 300,
    'positive_n': 76,
    'positive_rate': 0.25333333333333335,
    'mean_present_concepts': 10.22,
    'min_present_concepts': 4,
    'max_present_concepts': 18},
   'test': {'n': 400,
    'positive_n': 101,
    'positive_rate': 0.2525,
    'mean_present_concepts': 10.4825,
    'min_present_concepts': 5,
    'max_present_concepts': 19}},
  'icu_mortality': {'train': {'n': 1800,
    'positive_n': 94,
    'positive_rate': 0.052222222222222225,
    'mean_present_concepts': 9.901666666666667,
    'min_present_concepts': 4,
    'max_present_concepts': 21},
   'dev': {'n': 500,
    'positive_n': 26,
    'positive_rate': 0.052,
    'mean_present_concepts': 9.906,
    'min_pr

## Example samples

Each sample has the same schema as the clinical data: `txt`, binary `label`, and a list of present `[concept text, ICD-10 category]` pairs.

In [4]:
chest_train = json.loads(
    (DATA_DIR / "chest_trauma" / "train_samples.json").read_text()
)
mortality_train = json.loads(
    (DATA_DIR / "icu_mortality" / "train_samples.json").read_text()
)

chest_train[0], mortality_train[0]

({'txt': 'ed_note multiple_rib_fractures background_4 early_trauma_complication clavicle_scapula_fracture background_27 background_12 dx_A15 dx_O01 radiology_report dx_D57 minor_chest_wall_injury clavicle_scapula_fracture dx_X44 dx_M15 unspecified_thoracic_injury dx_A60 background_1 background_40 hypertension copd early_trauma_complication background_45 background_31 chronic_kidney_disease pulmonary_contusion type_2_diabetes',
  'concepts': [['Neutral synthetic ICD-10-category distractor A15', 'A15'],
   ['Neutral synthetic ICD-10-category distractor A60', 'A60'],
   ['Neutral synthetic ICD-10-category distractor D57', 'D57'],
   ['Type 2 diabetes mellitus', 'E11'],
   ['Essential hypertension', 'I10'],
   ['Other chronic obstructive pulmonary disease', 'J44'],
   ['Neutral synthetic ICD-10-category distractor M15', 'M15'],
   ['Chronic kidney disease', 'N18'],
   ['Neutral synthetic ICD-10-category distractor O01', 'O01'],
   ['Superficial injury of thorax', 'S20'],
   ['Fracture of r

## Experiment results

In [5]:
summary = pd.read_csv(DATA_DIR / "summary_results.csv")
summary[[
    "task_display", "variant",
    "concept_micro_aupr_mean", "concept_micro_aupr_std",
    "task_concept_macro_aupr_mean", "task_concept_macro_aupr_std",
    "concept_micro_f1_mean", "concept_micro_f1_std",
    "outcome_auroc_mean", "outcome_auroc_std",
    "outcome_aupr_mean", "outcome_aupr_std",
]].round(3)

,task_display,variant,concept_micro_aupr_mean,concept_micro_aupr_std,task_concept_macro_aupr_mean,task_concept_macro_aupr_std,concept_micro_f1_mean,concept_micro_f1_std,outcome_auroc_mean,outcome_auroc_std,outcome_aupr_mean,outcome_aupr_std
0,Chest trauma severity stratification,note_supervised,0.850,0.032,0.992,0.009,0.770,0.030,0.885,0.008,0.734,0.021
1,Chest trauma severity stratification,outcome_only,0.088,0.015,0.330,0.021,0.211,0.023,0.887,0.010,0.736,0.023
2,ICU in-hospital mortality prediction,note_supervised,0.794,0.017,0.988,0.008,0.718,0.018,0.885,0.005,0.493,0.040
3,ICU in-hospital mortality prediction,outcome_only,0.060,0.012,0.240,0.015,0.160,0.022,0.911,0.005,0.567,0.021


In [6]:
per_seed = pd.read_csv(DATA_DIR / "per_seed_results.csv")
per_seed[[
    "task_display", "variant", "model_seed",
    "concept_micro_aupr", "task_concept_macro_aupr",
    "concept_micro_f1", "outcome_auroc", "outcome_aupr",
]].sort_values(["task_display", "model_seed", "variant"]).round(3)

,task_display,variant,model_seed,concept_micro_aupr,task_concept_macro_aupr,concept_micro_f1,outcome_auroc,outcome_aupr
0,Chest trauma severity stratification,note_supervised,0,0.829,0.982,0.753,0.890,0.755
1,Chest trauma severity stratification,outcome_only,0,0.104,0.326,0.237,0.898,0.761
2,Chest trauma severity stratification,note_supervised,1,0.834,0.994,0.753,0.888,0.713
3,Chest trauma severity stratification,outcome_only,1,0.086,0.353,0.197,0.879,0.732
4,Chest trauma severity stratification,note_supervised,2,0.887,1.000,0.805,0.876,0.735
5,Chest trauma severity stratification,outcome_only,2,0.074,0.311,0.198,0.884,0.715
6,ICU in-hospital mortality prediction,note_supervised,0,0.786,0.993,0.703,0.890,0.523
7,ICU in-hospital mortality prediction,outcome_only,0,0.046,0.251,0.134,0.910,0.570
8,ICU in-hospital mortality prediction,note_supervised,1,0.783,0.979,0.713,0.880,0.448
9,ICU in-hospital mortality prediction,outcome_only,1,0.065,0.246,0.173,0.907,0.586


The mortality generator deliberately includes physiologic tokens such as elevated lactate and vasopressor requirement that have no ICD concept target. This makes mortality more heterogeneous and tests the limitation of an ICD-only bottleneck, rather than assuming that every predictive signal is representable by a diagnosis category.